# Milestone 3 — Inventory Optimization
**Group 2 · MIA5130A Systems Optimization and Management**

This notebook implements the inventory stage of the course project using the **same ten CA_1 food products** and the **validation-selected ridge forecasting model** from Milestone 2. It compares the instructor's two 100-unit scenarios: strict demand-proportional stocking and a revenue-oriented allocation that remains approximately proportional.

**One solution method throughout:** an analytical integer apportionment procedure, implemented in `Milestone3_Reproduce.py`. There is no external optimization solver. The full mathematical formulation and interpretation are in `Milestone3_Inventory_Optimization_Submission.pdf`.

## 1. Input files and experiment design

The input `weekly_forecasting_panel.csv` is the final Milestone 2 modeling panel (2,193 product-weeks; ten products, CA_1). The `calendar.csv` and `sell_prices.csv` files are the original M5 tables. The two **original Milestone 3** scenario CSVs are used **only for an explicit audit comparison**, because their forecasts/prices were undated.

The ridge model is reconstructed exactly using the training set through March 1, 2015, alpha = 100, population-standardized numeric predictors, product indicators, and an unpenalized intercept. Its preserved coefficients are used to forecast May 23–29, 2016, from information available by May 22. Validation and held-out test labels are never used to fit the model. The reconstruction verifies the earlier reported test metrics.

**To run:** put this notebook and `Milestone3_Reproduce.py` alongside the input CSVs (or change `DATA_DIR` below). Requires NumPy; plots are reproduced in the supplementary materials.

In [1]:
from pathlib import Path
from IPython.display import display, HTML, Image
from Milestone3_Reproduce import main
DATA_DIR = Path('.').resolve()  # Edit to point to your uploaded M5 CSV folder
OUTPUT_DIR = DATA_DIR / 'Milestone3_Support'
print('Source folder:', DATA_DIR)
print('Output folder:', OUTPUT_DIR)

Source folder: /mnt/data
Output folder: /mnt/data/Milestone3_Support


## 2. Reproduce the Milestone 2 handoff

The original selected ridge model is held fixed. The expected numerical check is test MAE approximately 25.118 and RMSE approximately 36.512 on 560 held-out product-weeks. The future forecast is based on past-only lags, the known calendar, and prices from the latest fully completed retail week ending May 20, 2016.

In [2]:
summary, forecast, scenario1, scenario2, steps = main(DATA_DIR, OUTPUT_DIR)
assert summary['m2_reproduced_check']['test_observations']==560
assert abs(summary['m2_reproduced_check']['test_MAE']-25.117915977858836)<1e-6
assert abs(summary['m2_reproduced_check']['test_RMSE']-36.512154705449625)<1e-6
print('Reproduction checks passed.')

{
  "method": "Analytical largest-remainder apportionment + price-ordered bounded integer allocation",
  "pricing_currency": "USD",
  "store_id": "CA_1",
  "forecast_origin": "2016-05-22",
  "forecast_start": "2016-05-23",
  "forecast_end": "2016-05-29",
  "forecasting_model": "Milestone 2 lag-based ridge regression, alpha=100; fit ends 2015-03-01",
  "m2_reproduced_check": {
    "test_observations": 560,
    "test_MAE": 25.117915977858836,
    "test_RMSE": 36.512154705449625,
    "train_rows": 1553,
    "train_cutoff": "2015-03-01",
    "ridge_alpha": 100.0
  },
  "forecast_total_units": 1550.8053078570158,
  "capacity_units": 100,
  "proportionality_tolerance_fraction": 0.2,
  "scenario1_allocated_units": 100,
  "scenario1_revenue_usd": 177.72,
  "scenario2_allocated_units": 100,
  "scenario2_revenue_usd": 188.18,
  "revenue_increase_usd": 10.46,
  "revenue_increase_pct": 5.885662840423134,
  "scenario1_total_abs_quota_deviation": 3.2858282049974123,
  "scenario2_total_abs_quota_devi

## 3. Scenario 1 — proportional policy

For each product *i*, define the exact quota \(q_i = 100d_i / \sum_j d_j\), where \(d_i\) is the unrounded forecast. The largest-remainder rule allocates each product its quota floor and awards the remaining integer units to the largest remainders. It minimizes total deviation from exact proportional shares subject to floor/ceiling rounding; revenue is evaluated afterward. Common constraints are total inventory of exactly 100, nonnegative integers, and allocation no greater than forecast demand.

## 4. Scenario 2 — price-prioritized policy

The objective is \(\max \sum_i p_i x_i\), where \(p_i\) is the assumed selling price. Common constraints still apply, plus \(\lfloor0.8q_ifloor \le x_i \le \lceil1.2q_iceil\). The 20% tolerance is explicitly **our policy assumption**, not a supplied data field or instructor threshold. Starting at all lower bounds, assign remaining units by descending selling price. This is the global revenue optimum **within the defined bounds** because each unit has equal capacity cost and the objective is linear.

In [3]:
def show_table(records, cols):
    head=''.join(f'<th style="padding:6px 8px;background:#e9eef4;text-align:left">{c}</th>' for c in cols)
    rows=''.join('<tr>'+''.join(f'<td style="padding:5px 8px;border-bottom:1px solid #ddd">{r[c]}</td>' for c in cols)+'</tr>' for r in records)
    display(HTML('<table><thead><tr>'+head+'</tr></thead><tbody>'+rows+'</tbody></table>'))
compare=[]
for a,b in zip(scenario1,scenario2):
    compare.append({'Product': a['Product_ID'], 'Forecast': f"{float(a['Forecasted_Demand']):.1f}",
                    'Price': f"${float(a['sell_price_USD']):.2f}",
                    'Scenario 1': a['Allocated_Inventory'], 'Scenario 2': b['Allocated_Inventory'],
                    'Revenue S1': a['Revenue_USD'], 'Revenue S2': b['Revenue_USD']})
show_table(compare,['Product','Forecast','Price','Scenario 1','Scenario 2','Revenue S1','Revenue S2'])
print(f"Revenue: Scenario 1 ${summary['scenario1_revenue_usd']:.2f}; Scenario 2 ${summary['scenario2_revenue_usd']:.2f}; gain ${summary['revenue_increase_usd']:.2f} ({summary['revenue_increase_pct']:.2f}%).")

Product,Forecast,Price,Scenario 1,Scenario 2,Revenue S1,Revenue S2
FOODS_3_586,273.1,$1.68,17,22,28.56,36.96
FOODS_3_252,259.0,$1.58,17,13,26.86,20.54
FOODS_3_714,125.0,$1.58,8,6,12.64,9.48
FOODS_3_607,102.5,$2.48,7,8,17.36,19.84
FOODS_3_064,179.8,$1.18,11,9,12.98,10.62
FOODS_3_555,134.4,$1.68,9,11,15.12,18.48
FOODS_2_197,133.9,$3.42,9,11,30.78,37.62
FOODS_3_739,83.3,$1.50,5,4,7.50,6.00
FOODS_3_501,153.9,$0.80,10,7,8.00,5.60
FOODS_3_099,105.8,$2.56,7,9,17.92,23.04


Revenue: Scenario 1 $177.72; Scenario 2 $188.18; gain $10.46 (5.89%).


## 5. Analytical solution evidence and feasibility checks

The solution begins with 75 units at minimum bounds and assigns the remaining 25 to higher-price products in five documented steps. Because a one-unit exchange to a higher-price product would increase revenue, an allocation with no remaining improving exchange is optimal within the chosen integer bounds. Both scenarios use exactly 100 units and stock no more than the forecast for any product.

In [4]:
show_table(steps,['sequence','product_id','price_usd','minimum_units','added_units','remaining_capacity_after_step','incremental_revenue_usd'])
assert summary['scenario1_allocated_units']==summary['scenario2_allocated_units']==100
assert summary['all_units_below_forecast']
assert summary['scenario2_revenue_usd'] > summary['scenario1_revenue_usd']
print('All inventory and optimality checks passed.')

sequence,product_id,price_usd,minimum_units,added_units,remaining_capacity_after_step,incremental_revenue_usd
1,FOODS_2_197,3.42,6,5,20,17.10
2,FOODS_3_099,2.56,5,4,16,10.24
3,FOODS_3_607,2.48,5,3,13,7.44
4,FOODS_3_555,1.68,6,5,8,8.40
5,FOODS_3_586,1.68,14,8,0,13.44


All inventory and optimality checks passed.


## 6. Original-file audit and interpretation

The original draft scenario files produced $206.40 and $254.74, but did not specify forecast origin or price provenance. The reproduced analysis replaces their undated inputs with a dated model-based forecast and last known completed-week prices. The original Scenario 2 used demand multiplied by price as an allocation weight; that is a useful heuristic but is **not a proof of optimal revenue**. The revised Scenario 2 has an explicit mathematical objective and feasible bounds.

The increase reported here is **potential sales revenue**, not realized revenue or net profit. M5 does not report purchase costs, lead times, unmet demand, delivery distances, or actual target-week sales after May 22, 2016. The 100-unit scenario is a course simplification, and the forecast may be wrong. Those caveats are discussed in the written Milestone 3 submission.

In [5]:
audit = __import__('csv').DictReader(open(OUTPUT_DIR/'m3_original_vs_reproduced_audit.csv'))
show_table(list(audit)[:10], ['Product_ID','Original_Forecasted_Demand','Reproduced_Ridge_Forecasted_Demand','Original_Sell_Price_USD','AsOf_2016_05_20_Sell_Price_USD'])
print('Submission materials are saved in',OUTPUT_DIR)

Product_ID,Original_Forecasted_Demand,Reproduced_Ridge_Forecasted_Demand,Original_Sell_Price_USD,AsOf_2016_05_20_Sell_Price_USD
FOODS_3_586,320,273.082769,2.48,1.68
FOODS_3_252,315,258.983515,2.24,1.58
FOODS_3_714,198,125.031702,1.58,1.58
FOODS_3_607,165,102.549307,1.18,2.48
FOODS_3_064,152,179.849954,1.0,1.18
FOODS_3_555,140,134.396048,1.48,1.68
FOODS_2_197,75,133.916064,5.24,3.42
FOODS_3_739,70,83.344151,3.98,1.50
FOODS_3_501,65,153.857724,1.98,0.80
FOODS_3_099,60,105.794075,1.28,2.56


Submission materials are saved in /mnt/data/Milestone3_Support
